In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# Load the data generated by our script
df = pd.read_csv('../data/economic_raw.csv')
df.set_index('Date', inplace=True)

def run_adf_test(series, name):
    """
    Null Hypothesis (H0): The series has a unit root (is non-stationary).
    Alternate Hypothesis (H1): The series is stationary.
    If p-value < 0.05, we reject H0 and conclude it is stationary.
    """
    result = adfuller(series.dropna())
    p_value = result[1]
    is_stationary = p_value < 0.05
    
    print(f"--- {name} ---")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value:       {p_value:.4f}")
    print(f"Stationary?    {'✅ YES' if is_stationary else '❌ NO'}\n")

print("TESTING RAW PRICES (Levels)\n")
for col in df.columns:
    run_adf_test(df[col], col)

print("="*40)
print("TESTING FIRST DIFFERENCES (Daily Change)\n")
df_diff = df.diff().dropna()
for col in df_diff.columns:
    run_adf_test(df_diff[col], f"{col} (Differenced)")

TESTING RAW PRICES (Levels)

--- Brent_Crude ---
ADF Statistic: -2.9478
p-value:       0.0401
Stationary?    ✅ YES

--- Gold ---
ADF Statistic: -1.3701
p-value:       0.5965
Stationary?    ❌ NO

--- USD_ILS ---
ADF Statistic: 0.1433
p-value:       0.9689
Stationary?    ❌ NO

--- SP500 ---
ADF Statistic: -1.0899
p-value:       0.7191
Stationary?    ❌ NO

--- VIX ---
ADF Statistic: -1.5978
p-value:       0.4847
Stationary?    ❌ NO

TESTING FIRST DIFFERENCES (Daily Change)

--- Brent_Crude (Differenced) ---
ADF Statistic: -10.3603
p-value:       0.0000
Stationary?    ✅ YES

--- Gold (Differenced) ---
ADF Statistic: -9.7030
p-value:       0.0000
Stationary?    ✅ YES

--- USD_ILS (Differenced) ---
ADF Statistic: -9.8186
p-value:       0.0000
Stationary?    ✅ YES

--- SP500 (Differenced) ---
ADF Statistic: -1.3247
p-value:       0.6178
Stationary?    ❌ NO

--- VIX (Differenced) ---
ADF Statistic: -8.4837
p-value:       0.0000
Stationary?    ✅ YES



**1. Raw Prices (Levels) — Random Walk Confirmation**
As expected for financial time series, the raw asset prices are predominantly non-stationary ($I(1)$ processes). Gold, USD/ILS, and the S&P 500 failed to reject the null hypothesis. Brent Crude marginally passed (p=0.0401), likely because it is oscillating within a localized, high-variance "war premium" band rather than trending infinitely, but it is not structurally stationary.

**2. First Differences (Absolute Daily Change)**
Taking the absolute daily difference ($P_t - P_{t-1}$) successfully forced Brent Crude, Gold, USD/ILS, and the VIX into strict stationarity (p=0.0000). 

**3. The Equity Volatility Anomaly**
The S&P 500 failed the ADF test *even after first differencing* (p=0.6178). This occurs because absolute point changes in massive indices do not stabilize variance—a 50-point drop at 4,000 is mathematically different than a 50-point drop at 7,000. Furthermore, geopolitical shocks introduce sustained volatility clustering that simple differencing cannot strip out.

**Conclusion & Feature Engineering Strategy**
Because simple differencing is insufficient for the S&P 500, we will abandon it. Instead, we will transform all economic time series into **Logarithmic Returns** ($\ln(P_t / P_{t-1})$) during the feature engineering pipeline. This is the quantitative standard to normalize continuous compounding, stabilize variance across disparate price magnitudes, and force robust stationarity.

In [2]:

print("TESTING LOG RETURNS\n")

# Calculate logarithmic returns: ln(Pt / Pt-1)
# We use np.log (which is the natural logarithm in numpy)
df_log_returns = np.log(df / df.shift(1)).dropna()

# We expect all of these, including the S&P 500, to pass comfortably
for col in df_log_returns.columns:
    run_adf_test(df_log_returns[col], f"{col} (Log Returns)")

TESTING LOG RETURNS

--- Brent_Crude (Log Returns) ---
ADF Statistic: -10.1757
p-value:       0.0000
Stationary?    ✅ YES

--- Gold (Log Returns) ---
ADF Statistic: -9.6811
p-value:       0.0000
Stationary?    ✅ YES

--- USD_ILS (Log Returns) ---
ADF Statistic: -9.8528
p-value:       0.0000
Stationary?    ✅ YES

--- SP500 (Log Returns) ---
ADF Statistic: -1.3528
p-value:       0.6047
Stationary?    ❌ NO

--- VIX (Log Returns) ---
ADF Statistic: -8.6946
p-value:       0.0000
Stationary?    ✅ YES





**Log Returns Transformation Test**
To stabilize the variance of the non-stationary variables, a Logarithmic Returns transformation ($\ln(P_t / P_{t-1})$) was applied. 
* Brent Crude, Gold, USD/ILS, and the VIX successfully achieved strict stationarity (p=0.0000). 
* **The S&P 500 failed again (p=0.6047).**

**Analysis & Engineering Decision**
The failure of log returns to stationarize the S&P 500 indicates a severe, non-mean-reverting structural break in the equities market (consistent with the simulated outbreak of a major West Asian conflict). Because forcing a secondary transformation (like double-differencing) risks destroying the signal, we can try dropping the S&P 500 from the fearure matrix.

However, we must test different methods first and also check how important the S&P 500 index was



**The Problem**
The S&P 500 failed to achieve stationarity even after applying logarithmic returns ($p=0.6047$). This indicates a severe, non-mean-reverting structural break in the equities market caused by the simulated geopolitical shock. Standard integer differencing ($d=1$) risks "over-differencing," which destroys the underlying signal and leaves only uninterpretable white noise. 

**The Experimental Strategy**
Before permanently dropping the S&P 500 from the feature matrix, we will test two advanced signal processing techniques to see if we can recover a stationary, mathematically viable feature that still represents global wealth destruction:

1. **De-Trending via Hodrick-Prescott (HP) Filter:** Financial time series consist of a long-term trend and short-term cyclical shocks. By applying the HP Filter (with a high $\lambda$ penalty for daily data), we will attempt to strip away the non-stationary macro trend and isolate the **cyclical component**. If this cyclical shock component is stationary, it can serve as a proxy for the immediate market panic.

2. **Fractional Differencing:**
   Instead of differentiating by a full integer ($d=1$), we will differentiate by a fraction ($0 < d < 1$). This technique, common in quantitative finance, achieves stationarity while preserving the "memory" and structural information of the original time series. We will iterate through fractional values to find the minimum $d$ required to achieve a p-value < 0.05.

**Outcome Logic**
If either of these methods yields a stationary series, we will retain the S&P 500 in the model using that transformation. If both fail, the S&P 500 will be dropped, and the model will rely on the VIX to represent systemic market stress.

In [5]:

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.filters.hp_filter import hpfilter

# --- Custom Fractional Differencing Math (Corrected) ---
def get_weights(d, size):
    """Calculates the binomial weights for fractional differencing."""
    w = [1.]
    for k in range(1, size):
        w.append(-w[-1] / k * (d - k + 1))
    return np.array(w)

def frac_diff(series, d, thres=1e-5):
    """Applies fractional differencing to a pandas series."""
    w = get_weights(d, len(series))
    
    # Flatten strictly to 1D and filter weights below the threshold
    w = w.flatten()
    w_ = w[np.abs(w) > thres]
    
    diff_series = pd.Series(index=series.index, dtype=float)
    
    for i in range(len(w_) - 1, len(series)):
        # Extract the window and REVERSE it so the newest price aligns with w[0]
        window = series.iloc[i - len(w_) + 1 : i + 1].values[::-1]
        
        # np.dot on two 1D arrays returns a scalar, no index [0] needed
        diff_series.iloc[i] = np.dot(w_, window)
        
    return diff_series.dropna()

# --- 1. De-trending via Hodrick-Prescott (HP) Filter ---
print("TESTING HP FILTER (Cyclical Component)\n")

sp500_dropna = df['SP500'].dropna()
cycle, trend = hpfilter(sp500_dropna, lamb=100000)

result_hp = adfuller(cycle)
print(f"ADF Statistic: {result_hp[0]:.4f}")
print(f"p-value:       {result_hp[1]:.4f}")
print(f"Stationary?    {'✅ YES' if result_hp[1] < 0.05 else '❌ NO'}\n")

print("="*40)

# --- 2. Fractional Differencing ---
print("TESTING FRACTIONAL DIFFERENCING\n")

for d in np.arange(0.1, 1.1, 0.1):
    frac_diff_series = frac_diff(sp500_dropna, d=d)
    
    if len(frac_diff_series) > 10:
        result_frac = adfuller(frac_diff_series)
        p_val = result_frac[1]
        
        print(f"d = {d:.1f} | p-value: {p_val:.4f} | {'✅ Stationary' if p_val < 0.05 else '❌ NO'}")
        
        if p_val < 0.05:
            print(f"\nSuccess! The S&P 500 becomes stationary at d={d:.1f}")
            break

TESTING HP FILTER (Cyclical Component)

ADF Statistic: -2.3831
p-value:       0.1466
Stationary?    ❌ NO

TESTING FRACTIONAL DIFFERENCING

d = 1.0 | p-value: 0.6178 | ❌ NO


* **HP Filter (Cyclical Component):** Failed stationarity ($p=0.1466$). The cyclical shocks are too sustained to mean-revert.
* **Fractional Differencing:** Failed to achieve stationarity at any valid fractional interval ($p=0.6178$ at $d=1.0$).

**Conclusion:**
The equities market exhibits a permanent structural break during the conflict window. The S&P 500 cannot be rendered stationary without mathematically destroying the underlying signal. 

**Action:** The S&P 500 is officially dropped from the predictive feature matrix. 
1. The **VIX** (stationary at $p=0.0000$) will serve as the sole exogenous proxy for systemic global panic in the Lanchester and Monte Carlo models.
2. If a measure of absolute wealth destruction is required for downstream dashboard visualizations, we will calculate a bounded "Percentage Drawdown from Pre-War Peak" rather than feeding raw equity prices into the stochastic engine.